# Final Dataset Construction

This notebook combines all engineered features produced during the data preparation phase.

Sources:
- listings_features.csv
- review_features.csv

The resulting dataset will be used for machine learning models such as Decision Trees and Random Forests.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

Project root: /Users/antoniacordova/Data for business/airbnb-rome-analysis


## 1. Load Engineered Datasets

The datasets generated in Notebook 02 and Notebook 03 are loaded and prepared for integration.

- listings_features.csv
- review_features.csv

In [2]:
listings_features = pd.read_csv(
    "../data/listings_features.csv"
)

review_features = pd.read_csv(
    "../data/review_features.csv"
)

## 2. Dataset Inspection

Before merging the datasets, we verify their dimensions, structure, and merge keys.

In [3]:
print(listings_features.shape)
print(review_features.shape)

(33564, 60)
(32255, 4)


In [4]:
listings_features.head()

,id,price,latitude,longitude,property_type,room_type,accommodates,bedrooms,beds,bathrooms,...,neighbourhood_cleansed_VII San Giovanni/Cinecittà,neighbourhood_cleansed_VIII Appia Antica,neighbourhood_cleansed_X Ostia/Acilia,neighbourhood_cleansed_XI Arvalia/Portuense,neighbourhood_cleansed_XII Monte Verde,neighbourhood_cleansed_XIII Aurelia,neighbourhood_cleansed_XIV Monte Mario,neighbourhood_cleansed_XV Cassia/Flaminia,distance_to_colosseum,location_cluster
0,2737,57.0,41.871360,12.482150,Private Room,Private room,1,1.0,1.0,1.5,...,False,True,False,False,False,False,False,False,2.252726,3.0
1,11834,110.0,41.895447,12.491181,Apartment,Entire home/apt,2,1.0,1.0,1.0,...,False,False,False,False,False,False,False,False,0.588865,4.0
2,12398,124.0,41.925820,12.469280,Apartment,Entire home/apt,6,2.0,3.0,1.0,...,False,False,False,False,False,False,False,False,4.389669,1.0
3,19965,162.0,41.908230,12.452930,Apartment,Entire home/apt,5,2.0,3.0,1.0,...,False,False,False,False,False,False,False,False,3.824847,1.0
4,19967,150.0,41.908283,12.452617,Apartment,Entire home/apt,5,2.0,3.0,1.0,...,False,False,False,False,False,False,False,False,3.850102,1.0


In [5]:
review_features.head()

,listing_id,review_count,avg_review_length,latest_review_date
0,2737,5,51.800000,2015-05-08
1,11834,284,76.450704,2025-07-05
2,12398,85,84.658824,2025-08-01
3,19965,178,44.410112,2025-08-05
4,19967,46,33.673913,2024-07-19


### 2.1 Verify Merge Keys

The listings dataset uses the variable id as its primary identifier, while the review dataset uses listing_id.

We verify that both identifiers exist before performing the merge operation.

In [7]:
print("id" in listings_features.columns)
print("listing_id" in review_features.columns)

True
True


## 3. Merge Listings and Review Features

Review-derived variables are merged with the listings dataset using the Airbnb listing identifier.

A left join is used so that all listings remain in the final dataset, even if some properties have never received reviews.

In [9]:
final_df = listings_features.merge(
    review_features,
    left_on="id",
    right_on="listing_id",
    how="left"
)

#deletion of duplicate key
final_df.drop(
    columns=["listing_id"],
    inplace=True
)

In [22]:
print(final_df.shape)

(33564, 64)


## 4. Handle Missing Review Features
Listings without reviews do not appear in the reviews dataset.

For these cases:

- review_count is set to 0
- avg_review_length is set to 0

This indicates the absence of review activity rather than missing information.

In [10]:
final_df["review_count"] = (
    final_df["review_count"]
    .fillna(0)
)

final_df["avg_review_length"] = (
    final_df["avg_review_length"]
    .fillna(0)
)

## 5. Additional Review Recency Feature

The latest review date is transformed into a numerical variable representing the number of days since the most recent review.

Listings without reviews receive a large placeholder value (9999) to indicate very low review recency.

In [12]:
final_df["latest_review_date"] = pd.to_datetime(
    final_df["latest_review_date"]
)

In [13]:
reference_date = final_df["latest_review_date"].max()

final_df["days_since_latest_review"] = (
    reference_date - final_df["latest_review_date"]
).dt.days

In [14]:
final_df["days_since_latest_review"] = (
    final_df["days_since_latest_review"]
    .fillna(9999)
)

## 6. Final Dataset Validation

Before exporting the final dataset, we verify:

- dataset dimensions
- variable types
- remaining missing values

In [15]:
print(final_df.shape)

(33564, 64)


In [16]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33564 entries, 0 to 33563
Data columns (total 64 columns):
 #   Column                                             Non-Null Count  Dtype         
---  ------                                             --------------  -----         
 0   id                                                 33564 non-null  int64         
 1   price                                              33564 non-null  float64       
 2   latitude                                           33564 non-null  float64       
 3   longitude                                          33564 non-null  float64       
 4   property_type                                      33564 non-null  str           
 5   room_type                                          33564 non-null  str           
 6   accommodates                                       33564 non-null  int64         
 7   bedrooms                                           33564 non-null  float64       
 8   beds                       

In [17]:
final_df.isnull().sum().sort_values(
    ascending=False
).head(20)

last_review               4295
latest_review_date        4295
location_cluster          3406
id                           0
property_type                0
room_type                    0
latitude                     0
price                        0
beds                         0
bathrooms                    0
host_is_superhost            0
host_response_rate           0
host_acceptance_rate         0
review_scores_rating         0
accommodates                 0
longitude                    0
review_scores_location       0
review_scores_value          0
reviews_per_month            0
number_of_reviews            0
dtype: int64

## 7. Export Final Dataset

The final integrated dataset is exported and will serve as the input for the machine learning models developed in Notebook 06.

In [21]:
final_df.to_csv(
    "../data/final_dataset.csv",
    index=False
)

print("final_dataset.csv saved")
print(final_df.shape)

final_dataset.csv saved
(33564, 64)
